# 00 - Setup & Data Pipeline

**Goal:** Set up the environment, download the Plant Village dataset and create the train/val/test splits.

**Structure:**
1. Setup & Imports
2. Create directories
3. Download dataset
4. Organize by class
5. Create train/val/test split
6. Data exploration

## 1. Setup & Imports

In [1]:
import os
import sys
import shutil
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

sys.path.insert(0, str(Path.cwd().parent / 'src'))

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('✅ All imports successful!')
print(f'Working directory: {Path.cwd()}')

✅ All imports successful!
Working directory: /Users/marco/Documents/repos/ComputerVisionProject/notebooks


## 2. Create Directories

In [2]:
ROOT = Path.cwd().parent

dirs = [
    ROOT / 'data' / 'raw',
    ROOT / 'data' / 'processed',
    ROOT / 'results' / 'models' / 'v1_hog_svm',
    ROOT / 'results' / 'models' / 'v2_custom_cnn',
    ROOT / 'results' / 'models' / 'v3_transfer_learning',
    ROOT / 'results' / 'metrics',
    ROOT / 'results' / 'plots',
]

for d in dirs:
    d.mkdir(parents=True, exist_ok=True)
    print(f'✅ {d.relative_to(ROOT)}')

print('\n📁 All directories ready!')

✅ data/raw
✅ data/processed
✅ results/models/v1_hog_svm
✅ results/models/v2_custom_cnn
✅ results/models/v3_transfer_learning
✅ results/metrics
✅ results/plots

📁 All directories ready!


## 3. Download Plant Village Dataset

Download from Kaggle (recommended method). Make sure you have the Kaggle API configured (`~/.kaggle/kaggle.json`).

Alternatively you can download manually from: https://www.kaggle.com/datasets/emmarex/plantdisease

In [ ]:
raw_dir = ROOT / 'data' / 'raw'

existing_classes = [d for d in raw_dir.iterdir() if d.is_dir()] if raw_dir.exists() else []

if len(existing_classes) >= 38:
    print(f'✅ Dataset already present: {len(existing_classes)} classes found in {raw_dir}')
else:
    print('📥 Dataset not found. Downloading...')
    print(f'Destination: {raw_dir}')

    # Option 1: Kaggle API
    os.system(f'kaggle datasets download -d vipoooool/new-plant-diseases-dataset -p {raw_dir} --unzip')
    
    # Option 2: if kaggle is unavailable, uncomment and adapt
    # os.system(f'wget <URL> -O {raw_dir}/dataset.zip && unzip {raw_dir}/dataset.zip -d {raw_dir}')

    existing_classes = [d for d in raw_dir.iterdir() if d.is_dir()]
    print(f'✅ Download complete: {len(existing_classes)} classes')

## 4. Detect Dataset Structure

The Kaggle dataset `vipoooool/new-plant-diseases-dataset` extracts into a nested folder and already contains the `train/` and `valid/` splits. This cell locates them recursively.

In [ ]:
def get_images(d):
    """Return all jpg/JPG images in a directory (case-insensitive)."""
    imgs = []
    for ext in ('*.jpg', '*.JPG', '*.jpeg', '*.JPEG'):
        imgs.extend(d.glob(ext))
    return imgs

def find_split_dir(base, name):
    """Recursively find the first folder with that name containing class subfolders."""
    for d in base.rglob(name):
        if d.is_dir() and any(c.is_dir() for c in d.iterdir()):
            return d
    return None

raw_dir = ROOT / 'data' / 'raw'
train_src = find_split_dir(raw_dir, 'train')
valid_src = find_split_dir(raw_dir, 'valid') or find_split_dir(raw_dir, 'val')

if train_src:
    class_dirs = sorted([d for d in train_src.iterdir() if d.is_dir()])
    print(f'✅ train: {train_src.relative_to(ROOT)}')
    print(f'   Classes: {len(class_dirs)}')
    print(f'   Images: {sum(len(get_images(d)) for d in class_dirs):,}')
else:
    class_dirs = []
    print('❌ train folder not found')

if valid_src:
    print(f'✅ valid: {valid_src.relative_to(ROOT)}')
    print(f'   Images: {sum(len(get_images(d)) for d in valid_src.iterdir() if d.is_dir()):,}')
else:
    print('⚠️  valid folder not found')

print('\nClasses (train):')
for i, d in enumerate(class_dirs):
    print(f'  [{i:02d}] {d.name}: {len(get_images(d))}')

## 5. Create Train / Val / Test Split

Copies `train` → `data/processed/train/` and splits `valid` into `val` (50%) and `test` (50%).

In [ ]:
processed_dir = ROOT / 'data' / 'processed'

def copy_split(src_dir, dst_dir, desc='Copy'):
    total = 0
    for class_dir in tqdm(sorted([d for d in src_dir.iterdir() if d.is_dir()]), desc=desc):
        dst_class = dst_dir / class_dir.name
        dst_class.mkdir(parents=True, exist_ok=True)
        for img in get_images(class_dir):
            dst = dst_class / img.name
            if not dst.exists():
                shutil.copy2(img, dst)
            total += 1
    return total

def split_valid_into_val_test(valid_src, processed_dir, random_state=42):
    stats = {'val': 0, 'test': 0}
    for class_dir in tqdm(sorted([d for d in valid_src.iterdir() if d.is_dir()]), desc='Split val/test'):
        images = sorted(get_images(class_dir))
        if not images:
            continue
        if len(images) == 1:
            dst = processed_dir / 'val' / class_dir.name / images[0].name
            dst.parent.mkdir(parents=True, exist_ok=True)
            if not dst.exists():
                shutil.copy2(images[0], dst)
            stats['val'] += 1
            continue
        indices = np.arange(len(images))
        val_idx, test_idx = train_test_split(indices, test_size=0.5, random_state=random_state)
        for split, idx_list in [('val', val_idx), ('test', test_idx)]:
            dst_class = processed_dir / split / class_dir.name
            dst_class.mkdir(parents=True, exist_ok=True)
            for idx in idx_list:
                dst = dst_class / images[idx].name
                if not dst.exists():
                    shutil.copy2(images[idx], dst)
                stats[split] += 1
    return stats

# Check whether the split is already in place (>= 10k images in train)
already_done = (
    (processed_dir / 'train').exists() and
    len(list((processed_dir / 'train').iterdir())) >= 38 and
    sum(len(get_images(c)) for c in (processed_dir / 'train').iterdir() if c.is_dir()) > 10000
)

if already_done:
    print('✅ Split already present in data/processed/')
    stats = {s: sum(len(get_images(c)) for c in (processed_dir / s).iterdir() if c.is_dir())
             for s in ['train', 'val', 'test']}
else:
    print('🔄 Creating split...')
    for s in ['train', 'val', 'test']:
        sp = processed_dir / s
        if sp.exists():
            shutil.rmtree(sp)
    stats = {}
    stats['train'] = copy_split(train_src, processed_dir / 'train', desc='Copy train')
    if valid_src:
        stats.update(split_valid_into_val_test(valid_src, processed_dir))
    else:
        print('⚠️  valid not found')

total = sum(stats.values())
print(f'\n📊 Dataset Statistics:')
for split, n in stats.items():
    print(f'  {split.upper():5s}: {n:6,} images ({n/total*100:.1f}%)')
print(f'  TOTAL: {total:6,} images')

## 6. Data Exploration

In [ ]:
train_dir = processed_dir / 'train'

class_counts = {d.name: len(get_images(d)) for d in sorted(train_dir.iterdir()) if d.is_dir()}
classes = list(class_counts.keys())
counts = list(class_counts.values())

split_totals = {
    s: sum(len(get_images(c)) for c in (processed_dir / s).iterdir() if c.is_dir())
    for s in ['train', 'val', 'test']
}

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

axes[0].barh(classes, counts, color='steelblue')
axes[0].set_xlabel('Number of images (train)')
axes[0].set_title('Image distribution per class')
axes[0].tick_params(axis='y', labelsize=7)

colors = ['#4ECDC4', '#FF6B6B', '#45B7D1']
bars = axes[1].bar(split_totals.keys(), split_totals.values(), color=colors)
axes[1].set_title('Images per split')
axes[1].set_ylabel('Number of images')
for bar, v in zip(bars, split_totals.values()):
    axes[1].text(bar.get_x() + bar.get_width() / 2, v + 200, f'{v:,}', ha='center', fontweight='bold')

plt.tight_layout()
plot_path = ROOT / 'results' / 'plots' / '00_data_exploration.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Plot saved: {plot_path.relative_to(ROOT)}')

In [ ]:
from matplotlib.image import imread

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for ax, class_name in zip(axes.flatten(), classes[:8]):
    img_files = get_images(train_dir / class_name)
    if img_files:
        ax.imshow(imread(str(img_files[0])))
        parts = class_name.split('___')
        label = f"{parts[0]}\n{parts[1].replace('_', ' ')}" if len(parts) == 2 else class_name
        ax.set_title(label, fontsize=9)
    ax.axis('off')

plt.suptitle('Sample images per class', fontsize=13)
plt.tight_layout()
plt.show()

print(f'\n✅ Notebook 00 complete!')
print(f'   Classes: {len(classes)}')
print(f'   Total images: {sum(split_totals.values()):,}')
print(f'\n➡️  Continue with: 01_v1_hog_svm.ipynb')